# Heterogeneous Effects with a Causal Forest on Weighted Cells — Public Coverage and Employment

**Artifact class:** `causal_analysis`  ·  **Data mode:** `empirical`  ·  **Vintage:** ACS 2023 1-year PUMS (weighted cells only)

This notebook uses **real data only**. There is no synthetic generator and no fallback. It asks how the adjusted difference in employment between adults with and without public health coverage, the quantity notebook C05 estimates, **varies across the measured characteristics C05 conditions on**. It fits a causal forest (`econml`, an optional extra) on **weighted cells, not people**: the Census client returns cell counts and no person row, so the forest's unit is a covariate cell and a treatment arm. That choice, made by the owner before the plan was written, changes what the forest's honesty and intervals mean, and the notebook says how each time it matters.

The analysis was fixed in a written plan (`pap/Q02_pap.md`) committed alone before any retrieval. **It inherits C05's identification and C05's failed negative control**, so unless this notebook's own negative control passes, everything below describes how an adjusted association varies, not how an effect varies.

*Proprietary. SPDX-License-Identifier: LicenseRef-Proprietary*

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why a causal forest needs a *unit*, and what changes when the only data are weighted cells.
2. Fit a doubly robust causal forest with honest splitting and read a conditional average effect by covariate cell.
3. Test whether a forest's forecast of heterogeneity is calibrated with the best-linear-predictor regression, and say what the test could detect.
4. Separate the few heterogeneity tests fixed in advance, and corrected for multiplicity, from the exploratory ones a forest invites.

## Prerequisites

**Knowledge.** The idea of confounding and adjustment (notebook C05); conditional average effects; cross-fitting (fitting a nuisance model on some data and using it on the rest). **Software.** The `econml` extra: `pip install -e '.[notebooks,econml]'`. **Data access.** A Census API key, read from the environment or from the reader's own key file (see the README); it is never printed here.

## Data Provenance

| Item | Meaning | Role |
|---|---|---|
| `PUBCOV` | with public health coverage | treatment |
| `ESR` | employment status recode | outcome (employed or not) |
| `NATIVITY` | foreign born | negative-control outcome |
| `AGEP`, `SEX`, `SCHL`, `DIS`, `NP` | age, sex, schooling, disability, household size, each banded as in C05 | covariates |
| state expansion status | `data/policy/medicaid/expansion_dates.csv` | covariate |

Source: ACS 2023 1-year PUMS through `fetch_acs_pums`, which returns **weighted cells and record counts only**. The request is C05's, unchanged; each run's records and a run ledger are in `provenance/`, bound to the plan by its digest. Two small metadata lookups (the Census variable labels and the state code list) are plain web requests that are not on the ledger.

## Design

**Unit.** One row per *covariate cell and treatment arm*: the arm's employment rate is the outcome and its record count is the weight (the construction C05 uses). A cell with only one arm is dropped and the share of treated records it holds is printed. The forest's honest splitting, subsampling and bootstrap of little bags run over these rows, so **its intervals describe stability to resampling cells, not people**. Both arms of a covariate cell are kept together when the nuisance models are cross-fitted (the `groups` argument of `fit`), and nowhere else.

**Estimand.** The conditional average effect on employment given the covariate cell, `CATE(x)`, in percentage points, and its average over treated records, which is compared with C05's stratified ATT.

**The family fixed in advance (Holm at the level in the configuration cell, four tests).** (1) the calibration coefficient on the forest's forecast of heterogeneity; (2-4) three subgroup contrasts, expansion status, sex and disability, each tested by the difference between stratified ATTs. Everything else, CATE by other covariates, importances, the most and least favourable cells, and the refits with other settings, is **exploratory** and labelled.

**What counts as no heterogeneity.** The calibration coefficient's 95% interval contains zero; the notebook then prints what the design could have detected. **What counts as credible as heterogeneous effects.** The negative control passes, the calibration interval lies above zero, and the declared contrasts survive Holm. C05's negative control did not pass.

In [1]:
"""Configuration. Every analytic choice is named here, not inferred."""
from __future__ import annotations

import hashlib
import io
import json
import re
import sys
import time as clock
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from econml.dml import CausalForestDML
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from scipy import stats
from sklearn.model_selection import GroupKFold

_here = Path.cwd().resolve()
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "krl_tutorials_data").is_dir():
        sys.path.insert(0, str(_cand / "src"))
        REPO_ROOT = _cand
        break
else:
    raise SystemExit("Could not locate krl_tutorials_data from " + str(_here))

from krl_tutorials_data import provenance
from krl_tutorials_data.methods import cate, matching
from krl_tutorials_data.sources import DataAccessError, fetch_acs_pums
from krl_tutorials_data.uncertainty import holm_bonferroni

NB_DIR = REPO_ROOT / "projects" / "11-distribution-heterogeneity" / "Q02_heterogeneous_effects_causal_forest"
PLAN = NB_DIR / "pap" / "Q02_pap.md"
AMEND: list[Path] = []                                   # no amendment at the time of writing
PLAN_SHA256 = hashlib.sha256(PLAN.read_bytes()).hexdigest()
C05_DIR = REPO_ROOT / "projects" / "09-causal-identification" / "C05_public_coverage_matching"
provenance.use_ledger(NB_DIR / "provenance" / "_runs.jsonl", PLAN, *AMEND)   # every fetch below is appended to the run ledger


def stop(message: str) -> None:
    """Append a stopped line to the run ledger, then raise."""
    provenance.ledger_stop(message)
    raise SystemExit(message)


YEAR = 2023
EXPANDED_BY = "2022-12-31"
SEED = 20261030
FOLDS = 5
N_TREES, SUBFOREST, MAX_SAMPLES = 2000, 4, 0.45
LEAF = 5                                  # forest min_samples_leaf; the refits use 20 and 50
NUISANCE_TREES, NUISANCE_LEAF = 200, 10
ALPHA = 0.05
MIN_TREATED_KEPT = 0.90                   # plan section 6.4: stop below this
AGREE_PP = 3.0                            # plan section 6.3
TARGET = "rate"
KEY = ["AGEP", "SEX", "SCHL", "DIS", "NP", "expanded"]
ADDED = "exploratory (outside the pre-specified family)"


def fmt_p(p: float) -> str:
    """A p-value without printing a false exact zero."""
    return "< 0.0001" if p < 0.0001 else f"= {p:.4f}"

print(f"pre-analysis plan : {PLAN.relative_to(REPO_ROOT)}")
print(f"plan SHA-256      : {PLAN_SHA256}")
print(f"vintage           : ACS {YEAR} 1-year PUMS")
print(f"seed {SEED}; {FOLDS} group folds; forest {N_TREES} trees, honest, min_samples_leaf {LEAF}, max_samples {MAX_SAMPLES}")

pre-analysis plan : projects/11-distribution-heterogeneity/Q02_heterogeneous_effects_causal_forest/pap/Q02_pap.md
plan SHA-256      : 821e4e0aed2cd169956adc5238456525691584494b1360b581157510bd2e7536
vintage           : ACS 2023 1-year PUMS
seed 20261030; 5 group folds; forest 2000 trees, honest, min_samples_leaf 5, max_samples 0.45


### Which states count as expanded

As in C05: the state's expansion status comes from the repository's policy table, and a state that expanded during 2023 is neither: it is excluded and counted.

In [2]:
policy = pd.read_csv(REPO_ROOT / "data" / "policy" / "medicaid" / "expansion_dates.csv")
policy["group"] = np.select(
    [policy["status"] == "Not Adopted",
     (policy["status"] == "Adopted") & (policy["expansion_effective"] <= EXPANDED_BY)],
    ["not expanded", "expanded"], default="excluded: expanded during 2023",
)
print(policy["group"].value_counts().to_string())

FIPS_URL = "https://www2.census.gov/geo/docs/reference/state.txt"
reply = requests.get(FIPS_URL, timeout=60, headers={"User-Agent": "telosyn-notebooks/1.0"})
reply.raise_for_status()
codes = pd.read_csv(io.StringIO(reply.text), sep="|", dtype=str)
fips = dict(zip(codes["STATE_NAME"], codes["STATE"]))
groups = {name: sorted(fips[s] for s in policy.loc[policy["group"] == name, "state"])
          for name in ("expanded", "not expanded")}
print({name: len(v) for name, v in groups.items()}, "states pooled in each group")

group
expanded                          39
not expanded                      10
excluded: expanded during 2023     2


{'expanded': 39, 'not expanded': 10} states pooled in each group


### Retrieving the cells, and checking they are C05's cells

The request is C05's, unchanged: two pooled requests, one per expansion group, each restricted to adults aged 19 to 64 below 200% of the poverty level, banded, summed across states, with cells of fewer than 10 records withheld only after pooling. The digest of each group's rows is then compared with the digest C05 stored for the same request. If they differ, Census has changed the file and the notebook stops.

In [3]:
for attempt in range(4):
    try:
        published = requests.get(
            f"https://api.census.gov/data/{YEAR}/acs/acs1/pums/variables.json", timeout=180
        ).json()["variables"]
        break
    except requests.RequestException:
        clock.sleep(10 * (attempt + 1))
else:
    stop("The Census variables list could not be read.")

CODES = ("AGEP", "SEX", "SCHL", "DIS", "NP", "PUBCOV", "ESR", "NATIVITY", "POVPIP")
VARIABLES = {code: published[code]["label"] for code in CODES}
AGGREGATE = ["AGEP", "SEX", "SCHL", "DIS", "NP", "PUBCOV", "ESR", "NATIVITY"]
KEEP = {"AGEP": (19, 64), "POVPIP": (0, 199)}
BANDS = {
    "AGEP": [19, 25, 30, 35, 40, 45, 50, 55, 60, 65],
    "SCHL": [0, 16, 18, 21, 25],
    "NP": [1, 2, 3, 4, 6, 21],
    "ESR": [0, 1, 3, 4, 6, 7],
}

pulls, mine = [], {}
try:
    for name, state_codes in groups.items():
        frame, prov = fetch_acs_pums(
            year=YEAR, variables=VARIABLES, aggregate_by=AGGREGATE,
            geography=[f"state:{c}" for c in state_codes], keep=KEEP, bands=BANDS,
        )
        frame["expanded"] = int(name == "expanded")
        pulls.append(frame)
        mine[name.replace(" ", "_")] = prov.content_sha256
        prov.with_pap_chain(PLAN, *AMEND).write(NB_DIR / "provenance" / f"q02_pums{YEAR}_{name.replace(' ', '_')}.json")
        a = frame.attrs
        print(f"{name:13s}: {len(state_codes)} states; records read {a['records_read']:>9,}, kept {a['records_kept']:>8,}; "
              f"cells {len(frame):>5,}; withheld {a['suppressed_records']:>6,} records "
              f"({a['suppressed_records'] / a['records_kept']:.1%})")
except DataAccessError as exc:
    print("HALTED - real data could not be obtained.")
    print(f"  failure class : {type(exc).__name__}")
    print(f"  detail        : {exc}")
    stop(f"data access failed: {type(exc).__name__}")
raw_cells = pd.concat(pulls, ignore_index=True)

theirs = {}
for key in mine:
    rec = json.loads((C05_DIR / "provenance" / f"c05_pums{YEAR}_{key}.json").read_text(encoding="utf-8"))
    theirs[key] = (rec[0] if isinstance(rec, list) else rec)["content_sha256"]
same = {key: mine[key] == theirs[key] for key in mine}
for key in mine:
    print(f"   {key:13s} digest here {mine[key][:12]}, C05's {theirs[key][:12]}, identical: {same[key]}")
print(f"FALSIFICATION: data_identity statistic={sum(same.values())} of {len(same)} groups whose rows have the digest C05 stored verdict={'PASS' if all(same.values()) else 'FAIL'}")
if not all(same.values()):
    stop("the rows differ from C05's stored digest; Census may have changed the file; ask the owner")

expanded     : 39 states; records read   289,644, kept  289,644; cells 4,207; withheld 13,630 records (4.7%)


not expanded : 10 states; records read   127,033, kept  127,033; cells 2,566; withheld 13,973 records (11.0%)
   expanded      digest here 7d6422e2d79a, C05's 7d6422e2d79a, identical: True
   not_expanded  digest here 0c2b31295765, C05's 0c2b31295765, identical: True
FALSIFICATION: data_identity statistic=2 of 2 groups whose rows have the digest C05 stored verdict=PASS


### From cells to forest units

As in C05, one row per covariate cell and treatment arm. The covariate cell is the key (age band, sex, schooling band, disability, household size band, state expansion status); the arm's outcome is its employment rate and its weight is its record count. Employment is `ESR` 1 (at work); `ESR` 3 and 6 are not employed; `ESR` 4 and 5 are dropped and counted. The negative-control outcome is the share foreign born, built the same way.

In [4]:
cells = raw_cells[raw_cells["ESR"].isin([1, 3, 6])].copy()
dropped = raw_cells.loc[~raw_cells["ESR"].isin([1, 3, 6]), "n_records"].sum()
print(f"records in dropped employment categories (armed forces or not applicable): {int(dropped):,}")
cells["treated"] = (cells["PUBCOV"] == 1).astype(int)
cells["employed"] = (cells["ESR"] == 1).astype(int)
cells["foreign"] = (cells["NATIVITY"] == 2).astype(int)
assert cells["employed"].var() > 0, "the outcome must vary"
for flag, label in (("e", "employed"), ("f", "foreign")):
    cells[flag] = cells["n_records"] * cells[label]
cells["n"] = cells["n_records"]
by_group = cells.groupby(KEY + ["treated"])[["n", "e", "f"]].sum().reset_index()
wide = by_group.pivot(index=KEY, columns="treated", values=["n", "e", "f"]).fillna(0.0)
wide.columns = [f"{name}{int(flag)}" for name, flag in wide.columns]
wide = wide.reset_index()

emp = cate.cell_units(wide, KEY)
nat = cate.cell_units(wide.assign(e1=wide["f1"], e0=wide["f0"]), KEY)
units = emp.units.copy()
units_nat = nat.units.copy()
assert units[TARGET].var() > 0, "the forest's outcome must vary"
print(f"covariate cells        : {emp.cells_total:,}; with both arms present: {emp.cells_kept:,}; dropped {emp.cells_dropped:,}")
print(f"forest units           : {len(units):,} (a covariate cell and an arm each), weights are record counts")
print(f"records                : treated {int(units.loc[units.treated == 1, 'n'].sum()):,}, untreated {int(units.loc[units.treated == 0, 'n'].sum()):,} in the cells kept")
print(f"share of treated records in cells with both arms: {emp.treated_share_kept:.1%}")
XCOLS = KEY
X = units[XCOLS].to_numpy(dtype=float)
Y, T, W, G = (units[TARGET].to_numpy(dtype=float), units["treated"].to_numpy(dtype=float),
              units["n"].to_numpy(dtype=float), units["cell"].to_numpy())

records in dropped employment categories (armed forces or not applicable): 474
covariate cells        : 1,336; with both arms present: 1,063; dropped 273
forest units           : 2,126 (a covariate cell and an arm each), weights are record counts
records                : treated 171,705, untreated 202,740 in the cells kept
share of treated records in cells with both arms: 95.7%


In [5]:
"""The forest, its cross-fitted calibration, and the overlap check (plan sections 4 and 6.4)."""
def make_forest(leaf: int = LEAF) -> CausalForestDML:
    """The pre-specified forest; every random choice takes SEED."""
    return CausalForestDML(
        model_y=RandomForestRegressor(n_estimators=NUISANCE_TREES, min_samples_leaf=NUISANCE_LEAF, random_state=SEED, n_jobs=1),
        model_t=RandomForestClassifier(n_estimators=NUISANCE_TREES, min_samples_leaf=NUISANCE_LEAF, random_state=SEED, n_jobs=1),
        discrete_treatment=True, cv=GroupKFold(n_splits=FOLDS), n_estimators=N_TREES, min_samples_leaf=leaf,
        max_samples=MAX_SAMPLES, honest=True, subforest_size=SUBFOREST, inference=True, random_state=SEED, n_jobs=1)


def crossfit(units_: pd.DataFrame, outcome: str, leaf: int = LEAF):
    """Held-out CATE and held-out nuisance residuals, folds over covariate cells."""
    x = units_[XCOLS].to_numpy(dtype=float)
    y, t, w, g = (units_[outcome].to_numpy(dtype=float), units_["treated"].to_numpy(dtype=float),
                  units_["n"].to_numpy(dtype=float), units_["cell"].to_numpy())
    tau, y_res, t_res = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(y))
    for train, test in GroupKFold(n_splits=FOLDS).split(x, y, g):
        forest = make_forest(leaf)
        forest.fit(y[train], t[train], X=x[train], sample_weight=w[train], groups=g[train])
        tau[test] = forest.effect(x[test])
        m_y = RandomForestRegressor(n_estimators=NUISANCE_TREES, min_samples_leaf=NUISANCE_LEAF, random_state=SEED, n_jobs=1)
        m_t = RandomForestClassifier(n_estimators=NUISANCE_TREES, min_samples_leaf=NUISANCE_LEAF, random_state=SEED, n_jobs=1)
        m_y.fit(x[train], y[train], sample_weight=w[train])
        m_t.fit(x[train], t[train].astype(int), sample_weight=w[train])
        y_res[test] = y[test] - m_y.predict(x[test])
        t_res[test] = t[test] - m_t.predict_proba(x[test])[:, 1]
    return tau, y_res, t_res


t0 = clock.time()
forest = make_forest()
forest.fit(Y, T, X=X, sample_weight=W, groups=G)
tau_all = forest.effect(X)
treated_rows = T == 1
att_forest = 100 * float(np.average(tau_all[treated_rows], weights=W[treated_rows]))
lo_u, hi_u = forest.ate_interval(X[treated_rows], alpha=ALPHA)
print(f"fitted the forest on {len(Y):,} units in {clock.time() - t0:.0f} s")
print(f"mean CATE over treated records (weights = record counts): {att_forest:+.2f} pp")
print(f"average CATE over the treated units, unweighted, with the forest's cell-level 95% interval: {100 * float(np.mean(tau_all[treated_rows])):+.2f} pp [{100 * lo_u:+.2f}, {100 * hi_u:+.2f}]")

tau_cf, y_res, t_res = crossfit(units, TARGET)
e_hat = T - t_res                                     # held-out propensity of each unit's arm indicator
print(f"held-out propensity range over units: {e_hat.min():.3f} to {e_hat.max():.3f}; spread (standard deviation) {e_hat.std():.3f}")
overlap_ok = emp.treated_share_kept >= MIN_TREATED_KEPT and e_hat.std() > 0.05
print(f"FALSIFICATION: overlap statistic={100 * emp.treated_share_kept:.1f} percent of treated records in cells with both arms; the plan stops below {100 * MIN_TREATED_KEPT:.0f} and below a held-out propensity spread of 0.05 verdict={'PASS' if overlap_ok else 'FAIL'}")
if not overlap_ok:
    stop("overlap failed the plan's rule; ask the owner")

fitted the forest on 2,126 units in 6 s
mean CATE over treated records (weights = record counts): -21.80 pp
average CATE over the treated units, unweighted, with the forest's cell-level 95% interval: -24.83 pp [-30.75, -18.91]


held-out propensity range over units: 0.129 to 0.890; spread (standard deviation) 0.198
FALSIFICATION: overlap statistic=95.7 percent of treated records in cells with both arms; the plan stops below 90 and below a held-out propensity spread of 0.05 verdict=PASS


### The calibration test: has the forest found heterogeneity it can calibrate?

Each unit's CATE is predicted by a forest trained on the other four folds (folds over covariate cells). The outcome minus its held-out prediction is regressed on the treatment residual times the mean forecast and times the forecast's deviation from that mean, weighted by record counts, with errors clustered by covariate cell. The **second coefficient is zero when the forest has found no heterogeneity and one when its forecast of heterogeneity is right.**

In [6]:
blp = cate.blp_calibration(y_res, t_res, tau_cf, weights=W, clusters=G, alpha=ALPHA)
names = ["mean forecast (1 = calibrated)", "heterogeneity forecast (0 = none found, 1 = right)"]
for k in range(2):
    print(f"   {names[k]:56s} {blp.coef[k]:+.3f}  (cluster-robust SE {blp.se[k]:.3f}; 95% interval {blp.lo[k]:+.3f} to {blp.hi[k]:+.3f})")
print(f"units {blp.n_units:,}; covariate-cell clusters {blp.n_clusters:,}")
p_blp = float(2 * stats.norm.sf(abs(blp.coef[1] / blp.se[1])))
detect = cate.detectable_coefficient(float(blp.se[1]))
print(f"two-sided p for the heterogeneity coefficient: p {fmt_p(p_blp)}")
print(f"the interval for the heterogeneity coefficient {'contains' if blp.lo[1] <= 1 <= blp.hi[1] else 'does not contain'} one, the value for a forecast of heterogeneity that is exactly right; a coefficient above one means the forecast understates the spread of the effects")
print(f"smallest heterogeneity coefficient this design detects with 80% power at the 5% level: {detect:.3f}")
print("reading: " + ("the interval for the heterogeneity coefficient lies above zero, so the forest's forecast carries calibrated heterogeneity"
                     if blp.lo[1] > 0 else "the interval for the heterogeneity coefficient contains or lies below zero, so no heterogeneity the forest found can be calibrated with this design's power"))

   mean forecast (1 = calibrated)                           +0.988  (cluster-robust SE 0.015; 95% interval +0.958 to +1.018)
   heterogeneity forecast (0 = none found, 1 = right)       +1.149  (cluster-robust SE 0.030; 95% interval +1.091 to +1.208)
units 2,126; covariate-cell clusters 1,063
two-sided p for the heterogeneity coefficient: p < 0.0001
the interval for the heterogeneity coefficient does not contain one, the value for a forecast of heterogeneity that is exactly right; a coefficient above one means the forecast understates the spread of the effects
smallest heterogeneity coefficient this design detects with 80% power at the 5% level: 0.084
reading: the interval for the heterogeneity coefficient lies above zero, so the forest's forecast carries calibrated heterogeneity


### The three subgroup contrasts fixed in advance, and the family correction

For each declared split the notebook computes C05's stratified ATT in each subgroup (`methods.matching.stratified_effect`, record counts) and tests the difference. Coding of the published variables is as recalled (`SEX` 1 male and 2 female; `DIS` 1 with and 2 without a disability) and is not verified in this build. The forest's mean CATE for each subgroup is printed beside it as a description. The four p-values, the calibration test's and the three contrasts', form the family, corrected by Holm.

In [7]:
def att(frame: pd.DataFrame):
    """Stratified ATT in pp and its standard error, and the share of treated records used."""
    r = matching.stratified_effect(frame["e1"], frame["n1"], frame["e0"], frame["n0"])
    return 100 * r.estimate, 100 * r.se, r.share_treated_used


SPLITS = [("expansion status", "expanded", 1, 0, "expanded", "not expanded"),
          ("sex", "SEX", 1, 2, "male", "female"),
          ("disability", "DIS", 1, 2, "with a disability", "without")]
treated_units = units.loc[treated_rows]
rows, p_values = [], [p_blp]
for label, col, a_val, b_val, a_name, b_name in SPLITS:
    ea, sa, ua = att(wide[wide[col] == a_val])
    eb, sb, ub = att(wide[wide[col] == b_val])
    diff, se = ea - eb, float(np.hypot(sa, sb))
    p = float(2 * stats.norm.sf(abs(diff / se)))
    fa = 100 * float(np.average(tau_all[treated_rows][treated_units[col].to_numpy() == a_val], weights=W[treated_rows][treated_units[col].to_numpy() == a_val]))
    fb = 100 * float(np.average(tau_all[treated_rows][treated_units[col].to_numpy() == b_val], weights=W[treated_rows][treated_units[col].to_numpy() == b_val]))
    rows.append({"split": label, "stratified ATT, first": ea, "second": eb, "difference": diff, "SE": se, "p": p,
                 "forest mean CATE, first": fa, "second ": fb})
    p_values.append(p)
    print(f"{label:17s}: {a_name} {ea:+.2f} pp, {b_name} {eb:+.2f} pp; difference {diff:+.2f} pp (SE {se:.2f}, 95% interval {diff - 1.96 * se:+.2f} to {diff + 1.96 * se:+.2f}, p {fmt_p(p)}); forest mean CATE {fa:+.2f} vs {fb:+.2f} pp")
corr = holm_bonferroni(np.array(p_values), alpha=ALPHA)
print()
print(corr.render())
fam = ["calibration (heterogeneity coefficient)", "expansion status", "sex", "disability"]
for name_, rej in zip(fam, corr.rejected):
    print(f"   {name_:42s} rejected after Holm: {bool(rej)}")

expansion status : expanded -21.06 pp, not expanded -24.41 pp; difference +3.35 pp (SE 0.40, 95% interval +2.56 to +4.14, p < 0.0001); forest mean CATE -21.33 vs -23.91 pp
sex              : male -25.67 pp, female -18.72 pp; difference -6.95 pp (SE 0.40, 95% interval -7.73 to -6.18, p < 0.0001); forest mean CATE -25.20 vs -19.29 pp
disability       : with a disability -32.66 pp, without -16.55 pp; difference -16.11 pp (SE 0.49, 95% interval -17.08 to -15.15, p < 0.0001); forest mean CATE -32.71 vs -16.72 pp

Holm-Bonferroni (FWER) at alpha = 0.05 over 4 tests: 4 significant uncorrected -> 4 after correction (0 withdrawn).
   calibration (heterogeneity coefficient)    rejected after Holm: True
   expansion status                           rejected after Holm: True
   sex                                        rejected after Holm: True
   disability                                 rejected after Holm: True


### The remaining pre-specified checks

The negative-control outcome (the share foreign born, which public coverage cannot change) is run through the same forest; the treated-record-weighted mean CATE is compared with C05's stratified ATT, read from C05's stored output.

In [8]:
forest_nc = make_forest()
Yn = units_nat[TARGET].to_numpy(dtype=float)
forest_nc.fit(Yn, T, X=X, sample_weight=W, groups=G)
tau_nc = forest_nc.effect(X)
lo_n, hi_n = forest_nc.ate_interval(X[treated_rows], alpha=ALPHA)
mean_nc = 100 * float(np.mean(tau_nc[treated_rows]))
nc_pass = 100 * lo_n <= 0 <= 100 * hi_n
print(f"negative control, share foreign born: unweighted average CATE over treated units {mean_nc:+.2f} pp, forest's cell-level 95% interval [{100 * lo_n:+.2f}, {100 * hi_n:+.2f}]")
print(f"FALSIFICATION: negative_control statistic={mean_nc:+.2f} pp average CATE on the share foreign born, interval [{100 * lo_n:+.2f}, {100 * hi_n:+.2f}] must contain zero verdict={'PASS' if nc_pass else 'FAIL'}")

negative control, share foreign born: unweighted average CATE over treated units -8.53 pp, forest's cell-level 95% interval [-12.85, -4.21]
FALSIFICATION: negative_control statistic=-8.53 pp average CATE on the share foreign born, interval [-12.85, -4.21] must contain zero verdict=FAIL


In [9]:
c05 = json.loads((C05_DIR / "C05_public_coverage_matching.ipynb").read_text(encoding="utf-8"))
c05_att = None
for cell_ in c05["cells"]:
    for out_ in cell_.get("outputs", []):
        for line in "".join(out_.get("text", [])).splitlines():
            m_ = re.search(r"stratified adjusted difference, record counts \(plan's headline; overlap cells\)\s+(-?\d+\.\d+)", line)
            if m_:
                c05_att = float(m_.group(1))
if c05_att is None:
    stop("C05's stored headline estimate could not be found")
mine_att, mine_se, mine_used = att(wide)
gap = att_forest - c05_att
agree = abs(gap) <= AGREE_PP
print(f"stratified ATT recomputed from these cells: {mine_att:+.2f} pp (SE {mine_se:.2f}); C05's stored headline: {c05_att:+.2f} pp")
print(f"forest mean CATE over treated records {att_forest:+.2f} pp; difference from C05's headline {gap:+.2f} pp")
print(f"FALSIFICATION: agreement_with_c05 statistic={gap:+.2f} pp between the forest's treated-record mean CATE and C05's stored stratified ATT; the plan allows {AGREE_PP:.0f} verdict={'PASS' if agree else 'FAIL'}")

stratified ATT recomputed from these cells: -21.67 pp (SE 0.19); C05's stored headline: -21.67 pp
forest mean CATE over treated records -21.80 pp; difference from C05's headline -0.13 pp
FALSIFICATION: agreement_with_c05 statistic=-0.13 pp between the forest's treated-record mean CATE and C05's stored stratified ATT; the plan allows 3 verdict=PASS


## Exploratory results

Everything from here to Limitations is **exploratory (outside the pre-specified family)**. A forest picks the cells it reports, so the most and least favourable cells below are selected by the forest and are not findings. No covariate is added, no band changed and no cell dropped on the strength of anything below.

In [10]:
print(f"{ADDED}: mean CATE over treated records by covariate level, forest fitted on all units")
base = treated_units.assign(tau=100 * tau_all[treated_rows])
for col in XCOLS:
    g_ = base.groupby(col).apply(lambda d: pd.Series({"treated records": d["n"].sum(), "mean CATE (pp)": np.average(d["tau"], weights=d["n"])}), include_groups=False)
    print(f"-- {col}")
    print(g_.to_string(float_format=lambda v: f"{v:,.1f}"))
imp = pd.Series(forest.feature_importances_, index=XCOLS).sort_values(ascending=False)
print(f"\n{ADDED}: the forest's variable importances (econml's feature_importances_)")
print(imp.round(3).to_string())
cellw = base.groupby("cell").apply(lambda d: pd.Series({"tau (pp)": np.average(d["tau"], weights=d["n"]), "treated records": d["n"].sum(), **{k: d[k].iloc[0] for k in XCOLS}}), include_groups=False)
big = cellw[cellw["treated records"] >= 100].sort_values("tau (pp)")
print(f"\n{ADDED}: the five most negative and five least negative covariate cells among those holding at least 100 treated records (selected by the forest)")
print(pd.concat([big.head(5), big.tail(5)]).to_string(float_format=lambda v: f"{v:,.1f}"))

exploratory (outside the pre-specified family): mean CATE over treated records by covariate level, forest fitted on all units
-- AGEP
      treated records  mean CATE (pp)
AGEP                                 
19.0         21,200.0           -16.0
25.0         16,073.0           -17.5
30.0         19,220.0           -17.2
35.0         18,279.0           -18.0
40.0         16,859.0           -20.0
45.0         13,943.0           -23.4
50.0         15,920.0           -27.3
55.0         21,079.0           -28.7
60.0         29,132.0           -26.0
-- SEX
     treated records  mean CATE (pp)
SEX                                 
1           72,810.0           -25.2
2           98,895.0           -19.3
-- SCHL
      treated records  mean CATE (pp)
SCHL                                 
0.0          31,278.0           -21.8
16.0         71,125.0           -23.9
18.0         52,285.0           -21.3
21.0         17,017.0           -14.5
-- DIS
     treated records  mean CATE (pp)
DIS          


exploratory (outside the pre-specified family): the five most negative and five least negative covariate cells among those holding at least 100 treated records (selected by the forest)
      tau (pp)  treated records  AGEP  SEX  SCHL  DIS  NP  expanded
cell                                                               
354      -44.2            120.0  35.0  1.0  16.0  1.0 1.0       0.0
473      -43.9            103.0  40.0  1.0  16.0  1.0 1.0       0.0
427      -43.9            154.0  35.0  2.0  18.0  1.0 1.0       1.0
311      -43.8            133.0  30.0  2.0  18.0  1.0 1.0       1.0
294      -43.1            200.0  30.0  2.0  16.0  1.0 1.0       1.0
455       -5.3            434.0  35.0  2.0  21.0  2.0 4.0       1.0
575       -5.3            365.0  40.0  2.0  21.0  2.0 4.0       1.0
577       -5.1            158.0  40.0  2.0  21.0  2.0 6.0       1.0
457       -5.1            159.0  35.0  2.0  21.0  2.0 6.0       1.0
454       -5.0            118.0  35.0  2.0  21.0  2.0 4.0       0.

In [11]:
print(f"{ADDED}: the same forest with other leaf sizes (the primary setting does not change)")
sens = []
for leaf in (20, 50):
    f_ = make_forest(leaf)
    f_.fit(Y, T, X=X, sample_weight=W, groups=G)
    t_ = f_.effect(X)
    tc, yr, tr_ = crossfit(units, TARGET, leaf)
    b_ = cate.blp_calibration(yr, tr_, tc, weights=W, clusters=G, alpha=ALPHA)
    sens.append({"min_samples_leaf": leaf, "mean CATE, treated records (pp)": 100 * float(np.average(t_[treated_rows], weights=W[treated_rows])),
                 "heterogeneity coefficient": b_.coef[1], "lower": b_.lo[1], "upper": b_.hi[1]})
sens.insert(0, {"min_samples_leaf": LEAF, "mean CATE, treated records (pp)": att_forest, "heterogeneity coefficient": blp.coef[1], "lower": blp.lo[1], "upper": blp.hi[1]})
print(pd.DataFrame(sens).to_string(index=False, float_format=lambda v: f"{v:+.3f}" if abs(v) < 5 else f"{v:+.2f}"))

exploratory (outside the pre-specified family): the same forest with other leaf sizes (the primary setting does not change)


 min_samples_leaf  mean CATE, treated records (pp)  heterogeneity coefficient  lower  upper
                5                           -21.80                     +1.149 +1.091 +1.208
               20                           -21.48                     +1.198 +1.124 +1.272
               50                           -21.35                     +1.330 +1.236 +1.423


In [12]:
"""The decision rule from the plan, applied to what was computed."""
hetero_found = blp.lo[1] > 0
survive = [bool(r) for r in corr.rejected]
credible = nc_pass and hetero_found and all(survive)
print(f"heterogeneity forecast calibrated (interval above zero): {hetero_found}")
print(f"negative control passes: {nc_pass}")
print(f"family members rejected after Holm: {dict(zip(fam, survive))}")
print("reading: " + ("credible as heterogeneous effects under the plan's rule" if credible else
      "NOT credible as heterogeneous effects under the plan's rule; reported as a description of how an adjusted association varies across cells"))
print(f"the treated-record mean CATE is {att_forest:+.2f} pp against C05's {c05_att:+.2f} pp, a gap of {gap:+.2f} pp; " + ("within the plan's allowance, so the forest and the stratified estimator agree on the average" if agree else "outside the plan's allowance, so the forest and the stratified estimator disagree on the average"))

heterogeneity forecast calibrated (interval above zero): True
negative control passes: False
family members rejected after Holm: {'calibration (heterogeneity coefficient)': True, 'expansion status': True, 'sex': True, 'disability': True}
reading: NOT credible as heterogeneous effects under the plan's rule; reported as a description of how an adjusted association varies across cells
the treated-record mean CATE is -21.80 pp against C05's -21.67 pp, a gap of -0.13 pp; within the plan's allowance, so the forest and the stratified estimator agree on the average


## Limitations

- **The unit is a cell and an arm, not a person.** The forest's honest splitting, subsampling and bootstrap of little bags run over covariate-cell and arm rows, which hold from a handful to thousands of records. Its intervals describe stability to resampling cells; they are not person-level intervals, and nothing here estimates an effect for an individual.
- **C05's identification is inherited, with its failed negative control.** Within a covariate cell, public coverage is assumed unrelated to employment potential; C05's negative control rejected that, and this notebook's own negative control is printed above. Unless it passes, the results describe how an adjusted association varies.
- **Inference ignores the survey design.** The client returns no replicate weights; the records are treated as the sample.
- **Coverage mixes programmes with different rules** (`PUBCOV` includes Medicare and military coverage), and coverage and employment are measured at the same date, so direction is not established.
- **Small cells are withheld** (the counts are printed at retrieval) and are not imputed; the covariate cells with only one arm are dropped and their share printed.
- **Discovered heterogeneity is exploratory** beyond the four tests fixed in advance; the forest chooses the cells it reports; the leaf-size refits show how much a cell's estimate depends on the forest's settings.
- **Departures from the pre-analysis plan.** The plan was not edited. Two readings of its wording are stated here: the plan's "forest's average effect" with an interval is read as the average of the forest's CATE over the treated units, unweighted, because the library offers no weighted-average interval (the treated-record-weighted mean is a point estimate, used for the agreement check); and the plan's calibration regression uses a held-out propensity and outcome prediction from plain random forests fitted on the same training folds as the held-out forest. The Census variable labels and the state code list are plain web requests, not on the run ledger. Coding of `SEX` and `DIS` is as recalled and not verified in this build.

## Key Takeaways

1. **On weighted cells the forest agrees with the stratified estimator on the average.** It ran on 2,126 units (1,063 covariate cells, each with two arms) that hold 95.7% of treated records, and its mean CATE over treated records is -21.80 pp against C05's -21.67 pp. The data are C05's: both groups' rows have the digest C05 stored.
2. **The forest found heterogeneity it can calibrate.** The heterogeneity coefficient is +1.149 with a 95% interval of +1.091 to +1.208 (the design could detect +0.084 or more), and the mean-forecast coefficient is +0.988. A heterogeneity coefficient above one says the forest's forecast *understates* the spread of the effects, not that it overstates it.
3. **All four tests fixed in advance survive the Holm correction.** The adjusted employment difference is -32.66 pp with a disability and -16.55 pp without (a difference of -16.11 pp), -25.67 pp for men and -18.72 pp for women (-6.95 pp), and -21.06 pp in expansion states against -24.41 pp in the others (+3.35 pp). The forest's own subgroup means order the groups the same way.
4. **The negative control fails, so this is a description of an adjusted association, not of how an effect varies.** The same forest finds an average CATE of -8.53 pp on the share foreign born, with an interval of -12.85 to -4.21, which public coverage cannot change. C05's version failed too. The heterogeneity is real in the sense that it is in the data and calibrated; what produces it is not identified.
5. **The unit is a cell, so the intervals are cell-level.** They say how stable the estimates are to resampling cells and arms, not people. Household size and disability carry nearly all of the forest's importance (0.640 and 0.274), and the leaf-size refits move the mean CATE by less than half a point (-21.80, -21.48 and -21.35 pp) while the heterogeneity coefficient rises with the leaf size (+1.149, +1.198, +1.330). All of that is exploratory.

## Exercises

1. Refit with `min_samples_leaf` 100 and say which of the exploratory cells keep their rank.
2. Add household size as a fourth declared split in a new amendment file, committed before you compute anything, and state what it does to the Holm correction.
3. Why can the forest's interval not be read as a person-level interval? Which part of the fitting would have to change for it to be one?

## References

Author, year and venue as recalled; **not verified against the source in this build.** Wager, S. and Athey, S. (2018), "Estimation and inference of heterogeneous treatment effects using random forests", *Journal of the American Statistical Association*. Athey, S., Tibshirani, J. and Wager, S. (2019), "Generalized random forests", *Annals of Statistics*. Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W. and Robins, J. (2018), "Double/debiased machine learning for treatment and structural parameters", *The Econometrics Journal*. Chernozhukov, V., Demirer, M., Duflo, E. and Fernandez-Val, I. (2018), "Generic machine learning inference on heterogeneous treatment effects in randomized experiments", NBER Working Paper.